In [ ]:
import subprocess
subprocess.run(['pip', 'install', '-q', 'rdflib', 'openai'], check=False)
print('Dependencies ready.')

In [ ]:
import json, os, re, time
from pathlib import Path
from datetime import datetime
from collections import defaultdict, Counter
from rdflib import Graph, URIRef, Literal, Namespace, RDF, OWL, RDFS
from openai import OpenAI
print('Imports loaded.')

In [ ]:
# -- Stage 5C FULL RUN configuration ----------------------------------------
BASE_DIR = Path('/Users/umair/Synthesising Regulatory Ontologies')

# INPUTS: Stage 5B full run output
STAGE5B_DIR        = BASE_DIR / '4 - Consolidation and Evaluation Layer' / 'output' / 'stage5b_alignment'
STAGE5B_MANIFEST   = STAGE5B_DIR / 'stage5b_alignment_manifest.json'
STAGE5B_TTL        = STAGE5B_DIR / 'cross_jurisdiction_alignments.ttl'

# OUTPUT: Stage 5C full run
OUTPUT_DIR         = BASE_DIR / '4 - Consolidation and Evaluation Layer' / 'output' / 'stage5c_shared_superclass'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SHARED_TTL         = OUTPUT_DIR / 'stage5c_shared_superclasses.ttl'
MANIFEST_PATH      = OUTPUT_DIR / 'stage5c_manifest.json'
RAW_LLM            = OUTPUT_DIR / 'raw_llm_naming.jsonl'
CHECKPOINT_DIR     = OUTPUT_DIR / 'checkpoints'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# Namespaces
NAMESPACES = {
    'cco':    'https://www.w3id.org/cco/cco#',
    'gro':    'https://w3id.org/cco-gro/onto#',
    'gro-uk': 'https://w3id.org/cco-gro/onto/uk#',
    'gro-us': 'https://w3id.org/cco-gro/onto/us#',
    'gro-ca': 'https://w3id.org/cco-gro/onto/ca#',
    'gro-au': 'https://w3id.org/cco-gro/onto/au#',
}

# Clustering configuration
MIN_CLUSTER_SIZE         = 2          # Pattern requires >=2 alignments
REQUIRE_SAME_PARENT      = True       # All cluster members must share CCO parent
INCLUDE_NARROW_BROAD     = False      # Only closeMatch + equivalentClass for clustering

# Naming convention
PATTERN_SUFFIX = 'Pattern'


from openai import OpenAI

# LLM configuration
LLM_MODEL    = 'Qwen/Qwen3-32B'
LLM_BASE_URL = 'https://inference.api.nscale.com/v1'
LLM_API_KEY  = ''  # enter you api key 
LLM_TEMPERATURE = 0.0
LLM_MAX_TOKENS  = 800

# Initialize client
client = OpenAI(
    api_key=LLM_API_KEY,
    base_url=LLM_BASE_URL,
)

print(f'Client initialized: {client is not None}')
print(f'API key set: {bool(LLM_API_KEY) and LLM_API_KEY != "your_actual_token_here"}')


# Checkpoint configuration
CHECKPOINT_INTERVAL = 5
RESUME_FROM_CHECKPOINT = True

# Verify configuration
if not LLM_API_KEY:
    raise RuntimeError('NSCALE_API_KEY not set. Run: export NSCALE_API_KEY="your_key"')

assert STAGE5B_MANIFEST.exists(), f'Stage 5B manifest not found: {STAGE5B_MANIFEST}'

print(f'Stage 5B manifest:    {STAGE5B_MANIFEST.name}')
print(f'Output dir:           {OUTPUT_DIR}')
print(f'LLM model:            {LLM_MODEL}')
print(f'Min cluster size:     {MIN_CLUSTER_SIZE}')
print(f'Same-parent required: {REQUIRE_SAME_PARENT}')
print(f'Pattern suffix:       {PATTERN_SUFFIX}')
print(f'Checkpoint interval:  {CHECKPOINT_INTERVAL}')

In [ ]:
# Quick API test
response = client.chat.completions.create(
    model=LLM_MODEL,
    messages=[{'role': 'user', 'content': 'Say hello'}],
    max_tokens=10,
)
print(f'API working. Response: {response.choices[0].message.content}')

In [ ]:
# -- Load Stage 5B manifest --------------------------------------------------
with open(STAGE5B_MANIFEST, encoding='utf-8') as f:
    stage5b = json.load(f)

validations = stage5b['validations']
print(f'Total Stage 5B validations: {len(validations)}')

aligned = [v for v in validations if v['decision'] == 'ALIGN']
print(f'Total ALIGN records: {len(aligned)}')

# Distribution
rel_dist = Counter(v.get('alignment_relation') for v in aligned)
print('\nAlignment relation distribution:')
for rel, n in rel_dist.most_common():
    print(f'  {rel}: {n}')

In [ ]:
# -- Form transitive clusters from skos:closeMatch pairs ---------------------
# Build adjacency from closeMatch pairs (and owl:equivalentClass if any)
adjacency = defaultdict(set)
class_metadata = {}  # class_qname -> {parent, jurisdiction, label, definition}

for v in aligned:
    rel = v.get('alignment_relation')
    
    # Skip narrow/broad for clustering (asymmetric relationships)
    if rel == 'skos:narrowMatch' or rel == 'skos:broadMatch':
        if not INCLUDE_NARROW_BROAD:
            continue
    
    a, b = v['class_a'], v['class_b']
    adjacency[a].add(b)
    adjacency[b].add(a)
    
    # Store metadata
    class_metadata[a] = {
        'jurisdiction': v['jurisdiction_a'],
        'parent':       v['parent'],
    }
    class_metadata[b] = {
        'jurisdiction': v['jurisdiction_b'],
        'parent':       v['parent'],
    }

# Lookup class labels/definitions from Stage 5A canonical
STAGE5A_CANONICAL = BASE_DIR / '4 - Consolidation and Evaluation Layer' / 'output' / 'stage5a_dedup' / 'stage5a_canonical_classes.json'
with open(STAGE5A_CANONICAL, encoding='utf-8') as f:
    canon = json.load(f)

# Index canonical by class qname
canon_idx = {}
for jur, classes in canon['per_jurisdiction'].items():
    for c in classes:
        canon_idx[c['class']] = c

# Enrich class_metadata with label and definition
for qname in class_metadata:
    if qname in canon_idx:
        class_metadata[qname]['label']      = canon_idx[qname]['label']
        class_metadata[qname]['definition'] = canon_idx[qname]['definition']
    else:
        class_metadata[qname]['label']      = qname.split(':')[-1]
        class_metadata[qname]['definition'] = ''

# BFS to find connected components
visited = set()
raw_clusters = []

for node in adjacency:
    if node in visited:
        continue
    cluster_members = set()
    queue = [node]
    while queue:
        current = queue.pop()
        if current in visited:
            continue
        visited.add(current)
        cluster_members.add(current)
        queue.extend(adjacency[current] - visited)
    
    if len(cluster_members) >= MIN_CLUSTER_SIZE:
        raw_clusters.append(cluster_members)

print(f'Raw connected components: {len(raw_clusters)}')

# Pre-condition validation
valid_clusters = []
rejected_clusters = []

for cluster_set in raw_clusters:
    members = [class_metadata[c] | {'class': c} for c in cluster_set]
    
    # Check 1: same CCO parent
    parents = set(m['parent'] for m in members)
    if REQUIRE_SAME_PARENT and len(parents) > 1:
        rejected_clusters.append({
            'reason': 'multiple_parents',
            'parents': list(parents),
            'members': [m['class'] for m in members],
        })
        continue
    
    # Check 2: different jurisdictions
    jurisdictions = set(m['jurisdiction'] for m in members)
    if len(jurisdictions) < 2:
        rejected_clusters.append({
            'reason': 'single_jurisdiction',
            'members': [m['class'] for m in members],
        })
        continue
    
    valid_clusters.append({
        'parent':        list(parents)[0],
        'size':          len(members),
        'jurisdictions': sorted(jurisdictions),
        'members':       members,
    })

# Assign IDs
for i, c in enumerate(valid_clusters, 1):
    c['cluster_id'] = i

print(f'\nValid clusters (>= {MIN_CLUSTER_SIZE} members, same parent, >=2 jurisdictions): {len(valid_clusters)}')
print(f'Rejected clusters: {len(rejected_clusters)}')

# Distribution
size_dist = Counter(c['size'] for c in valid_clusters)
print('\nCluster size distribution:')
for size, n in sorted(size_dist.items()):
    print(f'  {size} members: {n} cluster(s)')

parent_dist = Counter(c['parent'] for c in valid_clusters)
print('\nClusters by CCO parent:')
for p, n in parent_dist.most_common():
    print(f'  {p}: {n}')

# Show first 5 clusters as preview
print('\nFirst 5 clusters (preview):')
for c in valid_clusters[:5]:
    print(f'  Cluster {c["cluster_id"]} (parent: {c["parent"]}, size: {c["size"]}):')
    for m in c['members']:
        print(f'    {m["jurisdiction"]}: {m["class"]} — "{m["label"]}"')

In [ ]:
# -- Filter out mega-clusters (>10 members) ---------------------------------
MAX_CLUSTER_SIZE_FOR_PROCESSING = 10

valid_clusters_unfiltered = valid_clusters.copy()
valid_clusters = [c for c in valid_clusters_unfiltered if c['size'] <= MAX_CLUSTER_SIZE_FOR_PROCESSING]
skipped_for_size = [c for c in valid_clusters_unfiltered if c['size'] > MAX_CLUSTER_SIZE_FOR_PROCESSING]

# Re-assign cluster IDs after filtering
for i, c in enumerate(valid_clusters, 1):
    c['cluster_id'] = i

print(f'Cluster count before filter:  {len(valid_clusters_unfiltered)}')
print(f'Processable (size <= 10):     {len(valid_clusters)}')
print(f'Skipped (size > 10):          {len(skipped_for_size)}')
print()
print('Skipped mega-clusters (deferred for manual review):')
for c in skipped_for_size:
    print(f'  Size {c["size"]:3d} | parent: {c["parent"]} | jurisdictions: {c["jurisdictions"]}')

In [ ]:
# -- LLM prompt: naming + definition generation (PATTERN-INJECTED) -----------
LLM_SYSTEM_NAMING = """You are an ontology engineer naming a shared superclass for cross-jurisdiction equivalent classes under the Shared Domain Superclass Pattern.

PATTERN SPECIFICATION:

Trigger: At least two confirmed Functional Equivalence alignments with compatible CCO supertypes.

OWL Axioms (must hold):
1. The shared class must be a subclass of the confirmed CCO supertype. It cannot be introduced as a new root class above CCO.
2. Only axioms that hold universally across all aligned jurisdictions are placed on the shared class. Jurisdiction-specific axioms are retained at subclass level.
3. CCO disjointness axioms remain in force. A shared class cannot be typed as both Obligation and Permission — the AllDisjointClasses axiom forbids this.
4. Do not place on the shared class any axiom involving a CCO property whose domain or range constraint is not satisfied across all aligned jurisdictions.

Review Criteria:
- Label must NOT contain jurisdiction-specific vocabulary (no 'Apprenticeship', 'VET', 'HELP', 'Canada Student Loan', 'Pell Grant', 'Title IV', or equivalent jurisdiction-specific program names).
- Every axiom on the shared class must hold without exception across all aligned jurisdictions.
- The shared class must be a subclass of the confirmed CCO supertype — not a new root class.

Your task: given a cluster of functionally equivalent classes from different jurisdictions, produce:
1. A jurisdiction-neutral CamelCase name for the shared superclass
2. A jurisdiction-neutral human-readable label
3. A jurisdiction-neutral definition capturing the universal concept

Constraints:
- Name MUST be CamelCase, descriptive, jurisdiction-neutral
- Label is 2-4 words, human-readable
- Definition is 1-2 sentences, must NOT mention jurisdiction-specific programs
- Respond with valid JSON only. No prose, no markdown fences."""

LLM_USER_TEMPLATE_NAMING = """Generate a shared superclass for these functionally equivalent classes (Functional Equivalence verified):

CCO parent (confirmed shared supertype): {parent}

Cluster members:
{member_list}

Return this exact JSON shape:
{{
  "shared_class_name": "CamelCaseName",
  "label": "Human Readable Label",
  "definition": "Jurisdiction-neutral definition capturing the universal concept (1-2 sentences).",
  "naming_justification": "1-2 sentences explaining the chosen name and how it satisfies the pattern's review criteria."
}}"""

print('Pattern-injected prompt template defined.')

In [ ]:
# -- PATCHED HELPER (handles None responses) ---------------------------------

def _extract_first_json_object(text):
    if text is None or text == '':
        raise ValueError('Empty or None response from LLM')
    start = text.find('{')
    if start == -1:
        raise ValueError('No JSON object found')
    depth = 0; in_string = False; escape = False
    for i in range(start, len(text)):
        ch = text[i]
        if escape: escape = False; continue
        if ch == '\\': escape = True; continue
        if ch == '"': in_string = not in_string; continue
        if in_string: continue
        if ch == '{': depth += 1
        elif ch == '}':
            depth -= 1
            if depth == 0: return text[start:i+1]
    raise ValueError('Unbalanced braces')


def llm_generate_shared_name(cluster, retries=2):
    member_list_str = '\n'.join(
        f'- {m["jurisdiction"]}: {m["class"]} ({m["label"]}) — {m["definition"][:150]}'
        for m in cluster['members']
    )
    user_msg = LLM_USER_TEMPLATE_NAMING.format(
        parent=cluster['parent'],
        member_list=member_list_str,
    )
    
    last_err = None
    for attempt in range(retries + 1):
        try:
            resp = client.chat.completions.create(
                model=LLM_MODEL,
                messages=[
                    {'role': 'system', 'content': LLM_SYSTEM_NAMING},
                    {'role': 'user',   'content': user_msg},
                ],
                temperature=LLM_TEMPERATURE,
                max_tokens=LLM_MAX_TOKENS,
            )
            raw = resp.choices[0].message.content
            if raw is None or raw == '':
                raise ValueError('LLM returned empty response')
            parsed = json.loads(_extract_first_json_object(raw))
            return {'ok': True, 'parsed': parsed, 'raw': raw}
        except Exception as e:
            last_err = str(e)[:200]
            if attempt < retries:
                time.sleep(2 ** attempt)
    return {'ok': False, 'error': last_err}

print('PATCH APPLIED. Functions redefined.')

In [ ]:
# -- Combined: Resume logic + Generate names --------------------------------

# Resume from checkpoint if exists
cluster_names = []
raw_lines = []
start_idx = 0

if RESUME_FROM_CHECKPOINT:
    existing_checkpoints = sorted(CHECKPOINT_DIR.glob('checkpoint_*.json'))
    if existing_checkpoints:
        latest = existing_checkpoints[-1]
        print(f'Found existing checkpoint: {latest.name}')
        with open(latest, encoding='utf-8') as f:
            checkpoint = json.load(f)
        cluster_names = checkpoint['cluster_names']
        start_idx = checkpoint['last_idx']
        
        if RAW_LLM.exists():
            with open(RAW_LLM, encoding='utf-8') as f:
                raw_lines = [json.loads(line) for line in f]
        
        print(f'Resuming from cluster {start_idx + 1}/{len(valid_clusters)}')
    else:
        print('No checkpoint found, starting fresh')
else:
    print('Resume disabled, starting fresh')

# Generate shared superclass names per cluster
print(f'\nGenerating names for {len(valid_clusters) - start_idx} clusters (from {start_idx + 1})...')
print(f'Checkpoint will be saved every {CHECKPOINT_INTERVAL} clusters')
print('=' * 80)

for idx in range(start_idx, len(valid_clusters)):
    cluster = valid_clusters[idx]
    
    result = llm_generate_shared_name(cluster)
    
    if result['ok']:
        p = result['parsed']
        cluster_record = {
            'cluster_id':            cluster['cluster_id'],
            'parent':                cluster['parent'],
            'jurisdictions':         cluster['jurisdictions'],
            'size':                  cluster['size'],
            'members':               cluster['members'],
            'shared_class_name':     p.get('shared_class_name', f'SharedClass_{cluster["cluster_id"]}'),
            'label':                 p.get('label', f'Shared Class {cluster["cluster_id"]}'),
            'definition':            p.get('definition', ''),
            'naming_justification':  p.get('naming_justification', ''),
            'status':                'OK',
        }
        status_marker = '[OK]'
    else:
        cluster_record = {
            'cluster_id':            cluster['cluster_id'],
            'parent':                cluster['parent'],
            'jurisdictions':         cluster['jurisdictions'],
            'size':                  cluster['size'],
            'members':               cluster['members'],
            'shared_class_name':     f'SharedClass_{cluster["cluster_id"]}',
            'label':                 f'Shared Class {cluster["cluster_id"]}',
            'definition':            '',
            'naming_justification':  f'LLM_ERROR: {result["error"]}',
            'status':                'ERROR',
        }
        status_marker = '[ERROR]'
    
    cluster_names.append(cluster_record)
    raw_lines.append({
        'cluster_id': cluster['cluster_id'],
        'raw':        result.get('raw', ''),
        'error':      result.get('error', None),
    })
    
    name = cluster_record['shared_class_name']
    print(f'[{idx+1:4d}/{len(valid_clusters):4d}] Cluster {cluster["cluster_id"]} '
          f'({cluster["parent"]}, size {cluster["size"]}) -> gro:{name} {status_marker}')
    
    # Save checkpoint
    if (idx + 1) % CHECKPOINT_INTERVAL == 0:
        checkpoint_path = CHECKPOINT_DIR / f'checkpoint_{idx+1:05d}.json'
        with open(checkpoint_path, 'w', encoding='utf-8') as f:
            json.dump({
                'last_idx':       idx + 1,
                'total_clusters': len(valid_clusters),
                'cluster_names':  cluster_names,
                'progress_pct':   round((idx + 1) / len(valid_clusters) * 100, 1),
                'saved_at':       datetime.now().isoformat(),
            }, f, indent=2, ensure_ascii=False)
        
        with open(RAW_LLM, 'w', encoding='utf-8') as f:
            for line in raw_lines:
                f.write(json.dumps(line, ensure_ascii=False) + '\n')
        
        n_ok = sum(1 for c in cluster_names if c['status'] == 'OK')
        n_err = sum(1 for c in cluster_names if c['status'] == 'ERROR')
        print(f'  CHECKPOINT SAVED: {checkpoint_path.name} | OK={n_ok} ERROR={n_err}')

# Final save
with open(RAW_LLM, 'w', encoding='utf-8') as f:
    for line in raw_lines:
        f.write(json.dumps(line, ensure_ascii=False) + '\n')

n_ok = sum(1 for c in cluster_names if c['status'] == 'OK')
n_err = sum(1 for c in cluster_names if c['status'] == 'ERROR')
print('=' * 80)
print(f'COMPLETE: OK={n_ok} | ERROR={n_err}')

In [ ]:
# -- Build shared-superclass TTL graph ---------------------------------------
def qname_to_uri(qname):
    if ':' in qname:
        pfx, local = qname.split(':', 1)
        if pfx in NAMESPACES:
            return URIRef(NAMESPACES[pfx] + local)
    return URIRef(qname)

def apply_pattern_suffix(name: str) -> str:
    """Ensure the name ends with PATTERN_SUFFIX."""
    return name if name.endswith(PATTERN_SUFFIX) else f"{name}{PATTERN_SUFFIX}"

graph = Graph()
for pfx, uri in NAMESPACES.items():
    graph.bind(pfx, uri)
graph.bind('owl', OWL)
graph.bind('rdfs', RDFS)
SKOS_NS = Namespace('http://www.w3.org/2004/02/skos/core#')
graph.bind('skos', SKOS_NS)

# Track final names (handle duplicates)
final_names_used = set()

for record in cluster_names:
    original_name = record['shared_class_name']
    candidate_name = apply_pattern_suffix(original_name)
    
    # Handle name collisions (rare, but safe)
    final_name = candidate_name
    counter = 2
    while final_name in final_names_used:
        final_name = f'{candidate_name}_{counter}'
        counter += 1
    final_names_used.add(final_name)
    
    record['shared_class_name_llm_raw'] = original_name
    record['shared_class_name']         = final_name
    
    shared_uri = qname_to_uri(f'gro:{final_name}')
    parent_uri = qname_to_uri(record['parent'])
    
    # Shared superclass declaration
    graph.add((shared_uri, RDF.type, OWL.Class))
    graph.add((shared_uri, RDFS.subClassOf, parent_uri))
    graph.add((shared_uri, RDFS.label, Literal(record['label'])))
    
    # Definition (skos:definition)
    if record.get('definition'):
        graph.add((shared_uri, SKOS_NS.definition, Literal(record['definition'])))
    
    # Provenance via rdfs:comment
    if record.get('naming_justification'):
        graph.add((shared_uri, RDFS.comment, Literal(record['naming_justification'])))
    
    # Aligned classes become subclasses
    for m in record['members']:
        member_uri = qname_to_uri(m['class'])
        graph.add((member_uri, RDFS.subClassOf, shared_uri))

graph.serialize(destination=str(SHARED_TTL), format='turtle')
print(f'Shared-superclass TTL saved: {SHARED_TTL}')
print(f'  Total triples: {len(graph)}')

# Verify parse
try:
    verify = Graph()
    verify.parse(str(SHARED_TTL), format='turtle')
    print(f'  Parse verification: OK ({len(verify)} triples)')
except Exception as e:
    print(f'  Parse FAILED: {e}')

In [ ]:
# -- Save manifest -----------------------------------------------------------
manifest = {
    'metadata': {
        'run_type':              'stage5c_shared_superclass_full_run',
        'stage':                 'stage5c',
        'llm_model':             LLM_MODEL,
        'min_cluster_size':      MIN_CLUSTER_SIZE,
        'same_parent_required':  REQUIRE_SAME_PARENT,
        'pattern_suffix':        PATTERN_SUFFIX,
        'definition_policy':     'llm_generated_with_skos_definition',
        'odp_pattern':           'ODP-13 Shared Domain Superclass',
        'odp_precondition':      'ODP-12 Functional Equivalence (>= 2 aligned classes)',
        'created_at':            datetime.now().isoformat(),
        'stage5b_source':        str(STAGE5B_MANIFEST),
    },
    'totals': {
        'n_stage5b_alignments':       len(aligned),
        'n_raw_clusters':             len(raw_clusters),
        'n_valid_clusters':           len(valid_clusters),
        'n_rejected_clusters':        len(rejected_clusters),
        'n_shared_superclasses':      len(cluster_names),
        'n_llm_ok':                   n_ok,
        'n_llm_error':                n_err,
        'n_subclassof_triples':       sum(c['size'] for c in cluster_names),
        'n_total_triples':            len(graph),
    },
    'rejected_clusters':   rejected_clusters,
    'shared_superclasses': cluster_names,
}

with open(MANIFEST_PATH, 'w', encoding='utf-8') as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)

print(f'Manifest saved: {MANIFEST_PATH}')

In [ ]:
# -- Final summary -----------------------------------------------------------
print('=' * 70)
print('STAGE 5C FULL RUN SUMMARY')
print('=' * 70)
print(f'  Stage 5B alignments input:        {len(aligned)}')
print(f'  Raw clusters from closeMatch:     {len(raw_clusters)}')
print(f'  Valid clusters (after filter):    {len(valid_clusters)}')
print(f'  Mega-clusters skipped (>10):      {len(skipped_for_size)}')
print(f'    Retained after manual review:   2')
print(f'    Dropped (semantic incoherence): 1')
print(f'  Shared superclasses generated:    {len(cluster_names)}')

# Status breakdown (corrected to include manual statuses)
status_counts = Counter(c['status'] for c in cluster_names)
print(f'    LLM-generated:                  {status_counts.get("OK", 0)}')
print(f'    Manually named (small):         {status_counts.get("OK_MANUAL", 0)}')
print(f'    Manually named (mega):          {status_counts.get("OK_MANUAL_MEGA", 0)}')
print(f'    Manually named (theme split):   {status_counts.get("OK_MANUAL_THEME", 0)}')
print(f'    Errors:                         {status_counts.get("ERROR", 0)}')

print(f'  rdfs:subClassOf triples:          {sum(c["size"] for c in cluster_names)}')
print(f'  Total TTL triples:                {len(graph)}')
print()

# Cluster size distribution
print('Cluster size distribution:')
size_dist_final = Counter(c['size'] for c in cluster_names)
for size, n in sorted(size_dist_final.items()):
    print(f'  {size:3d} members: {n} cluster(s)')

# Per-parent distribution
print('\nShared superclasses by CCO parent:')
parent_dist_final = Counter(c['parent'] for c in cluster_names)
for p, n in parent_dist_final.most_common():
    print(f'  {p}: {n}')

# Per-jurisdiction participation
jur_part = Counter()
for c in cluster_names:
    for j in c['jurisdictions']:
        jur_part[j] += 1
print('\nPer-jurisdiction participation in shared superclasses:')
for j in sorted(jur_part):
    print(f'  {j}: {jur_part[j]}')

# Example superclasses
print('\nFirst 10 shared superclasses:')
for c in cluster_names[:10]:
    print(f'  gro:{c["shared_class_name"]}Pattern '
          f'(parent: {c["parent"]}, {c["size"]} members, {len(c["jurisdictions"])} jurisdictions)')

print(f'\nOutput files:')
print(f'  Shared TTL:     {SHARED_TTL.name}')
print(f'  Manifest:       {MANIFEST_PATH.name}')
print(f'  Raw LLM:        {RAW_LLM.name}')
print(f'  Checkpoints:    {CHECKPOINT_DIR}/')
print()
print('Next: Stage 5D — merge all TTLs + apply OWL reasoner + evaluation phase.')

In [ ]:
# Apply manual assignments to cluster_names + drop Cluster 1

CLUSTER_TO_DROP = 1  # Mega-cluster (120) — junk drawer

# First, find current cluster_names entries for our targets
# Note: After filter, cluster_ids were re-assigned. We need to match by content.

# But mega-clusters (1, 3, 13) were excluded by filter — they're in skipped_for_size
# Errored (2, 7) are in cluster_names with their current IDs

# Step 1: Apply manual names to errored clusters (2, 7) - already in cluster_names
errored_manual = {
    2: manual_assignments[2],
    7: manual_assignments[7],
}

for cluster_id, manual_data in errored_manual.items():
    for c in cluster_names:
        if c['cluster_id'] == cluster_id:
            c['shared_class_name']    = manual_data['name']
            c['label']                = manual_data['label']
            c['definition']           = manual_data['definition']
            c['naming_justification'] = f'Manually named due to persistent LLM empty responses across multiple retries.'
            c['status']               = 'OK_MANUAL'
            print(f'Cluster {cluster_id} updated: gro:{c["shared_class_name"]}')

# Step 2: Add manual entries for mega-clusters 3 and 13 (currently in skipped_for_size)
mega_to_keep = [3, 13]  # Coherent mega-clusters
next_id = max(c['cluster_id'] for c in cluster_names) + 1

for original_cluster_id in mega_to_keep:
    # Find the original mega-cluster in skipped_for_size
    original = next((c for c in skipped_for_size if c['cluster_id'] == original_cluster_id), None)
    if not original:
        # Try matching by parent + size if cluster_id reassigned
        if original_cluster_id == 3:
            original = next((c for c in skipped_for_size if c['size'] == 41), None)
        elif original_cluster_id == 13:
            original = next((c for c in skipped_for_size if c['size'] == 13), None)
    
    if not original:
        print(f'WARNING: Original cluster {original_cluster_id} not found in skipped_for_size')
        continue
    
    manual_data = manual_assignments[original_cluster_id]
    
    new_record = {
        'cluster_id':            next_id,
        'parent':                original['parent'],
        'jurisdictions':         original['jurisdictions'],
        'size':                  original['size'],
        'members':               original['members'],
        'shared_class_name':     manual_data['name'],
        'label':                 manual_data['label'],
        'definition':            manual_data['definition'],
        'naming_justification':  f'Manually named — large transitive cluster ({original["size"]} members) identified as semantically coherent shared concept after manual review.',
        'status':                'OK_MANUAL_MEGA',
    }
    cluster_names.append(new_record)
    print(f'Mega-cluster (size {original["size"]}) added: gro:{manual_data["name"]} as cluster_id={next_id}')
    next_id += 1

# Step 3: Cluster 1 (120 members) — DROP, no action needed (not in cluster_names)
print(f'\nCluster 1 (120 members): DROPPED — no shared superclass created')

# Final summary
print(f'\n{"="*60}')
print(f'Total shared superclasses now: {len(cluster_names)}')
print(f'  LLM OK:         {sum(1 for c in cluster_names if c["status"] == "OK")}')
print(f'  Manual (small): {sum(1 for c in cluster_names if c["status"] == "OK_MANUAL")}')
print(f'  Manual (mega):  {sum(1 for c in cluster_names if c["status"] == "OK_MANUAL_MEGA")}')
print(f'  Errors:         {sum(1 for c in cluster_names if c["status"] == "ERROR")}')

In [ ]:
print(f'cluster_names has {len(cluster_names)} entries')
print(f'Status breakdown:')
from collections import Counter
status_counts = Counter(c['status'] for c in cluster_names)
for s, n in status_counts.items():
    print(f'  {s}: {n}')

In [ ]:
# -- Apply 7-theme decomposition for Cluster 1 (120 members) -----------------

# Theme definitions with member assignments
THEME_DECOMPOSITION = {
    'FundingApplication': {
        'label': 'Funding Application',
        'definition': 'A formal application or submission for student financial assistance, loan rehabilitation, or other funding programs.',
        'members': [
            'gro-ca:FundingAndLoanRehabilitationApplication',
            'gro-uk:ApplicationAndSupportingEvidence',
            'gro-au:VETStudentLoanApplication',
            'gro-ca:PartTimeStudentLoanApplication',
            'gro-us:Application',
            'gro-us:FAFSAApplication',
            'gro-ca:Application',
            'gro-ca:StudentFinancialAssistanceApplication',
        ],
    },
    'EducationalProgram': {
        'label': 'Educational Program',
        'definition': 'A structured course of study or training program offered by an educational provider.',
        'members': [
            'gro-ca:BrokeredProgramOfStudy',
            'gro-uk:DfEFundedCourse',
            'gro-uk:Programme',
            'gro-ca:ExecutiveProgram',
            'gro-uk:ApprenticeshipProgramme',
            'gro-ca:Program',
            'gro-au:Course',
            'gro-au:DeterminationCourse',
        ],
    },
    'FinancialAidResource': {
        'label': 'Financial Aid Resource',
        'definition': 'A monetary or financial resource representing aid, funding, loans, grants, bursaries, or scholarships allocated to support students or educational programs.',
        'members': [
            'gro-ca:Bursary',
            'gro-us:LateDisbursement',
            'gro-au:FeeLimit',
            'gro-au:LoanAmount',
            'gro-ca:DependantAllowance',
            'gro-ca:StudentFinancialAssistance',
            'gro-ca:FundingAllocationResource',
            'gro-us:TitleIVAid',
            'gro-uk:MaximumFundingAmount',
            'gro-ca:Overaward',
            'gro-us:PellGrantMoney',
            'gro-uk:ExcessCostPayment',
            'gro-ca:FinancialAssistanceProgram',
            'gro-uk:AdvancedLearnerLoan',
            'gro-ca:StudentAward',
            'gro-us:FinancialResource',
            'gro-us:UndergraduateAid',
            'gro-us:StateFinancialAid',
            'gro-ca:ScholarshipBursaryResource',
            'gro-uk:TransferAmount',
            'gro-au:FundingAllocation',
            'gro-ca:TravelExpenseFunding',
            'gro-au:FinancialResource',
            'gro-ca:WorkBCEmploymentServicesFunds',
            'gro-ca:GrantFunding',
            'gro-us:DirectUnsubsidizedAndPLUSLoans',
            'gro-ca:FinancialResource',
            'gro-uk:BursaryResource',
            'gro-ca:RepaymentAssistancePlan',
            'gro-au:TuitionFeeBenefit',
            'gro-uk:StudentLoan',
            'gro-uk:FundingResource',
            'gro-ca:FundingProgram',
            'gro-ca:StudentFinancialAssistanceProgram',
            'gro-uk:AdditionalPayment',
            'gro-au:TuitionFeeLoanRefund',
            'gro-us:DirectLoanFunds',
            'gro-ca:AdultUpgradingGrant',
            'gro-ca:LivingAllowance',
            'gro-uk:FinancialResource',
            'gro-au:ProviderFeeLimitExcessLoanAmount',
            'gro-ca:AccessibilitySupportsFunding',
            'gro-ca:AviationProgramFunding',
            'gro-au:CourseFeeLoan',
            'gro-us:DirectUnsubsidizedLoan',
            'gro-uk:ProviderAdditionalPayment',
        ],
    },
    'EvidentiaryDocument': {
        'label': 'Evidentiary Document',
        'definition': 'A document, record, or information resource serving as evidence or documentation in regulatory or eligibility contexts.',
        'members': [
            'gro-au:StudentData',
            'gro-au:VSLInformation',
            'gro-au:Information',
            'gro-au:InformationOrDocument',
            'gro-ca:StudentFinancialAssistanceInformation',
            'gro-au:InformationResource',
            'gro-uk:EvidencePack',
            'gro-us:Document',
            'gro-uk:Document',
            'gro-us:DocumentationItem',
            'gro-uk:EvidenceRequirement',
            'gro-ca:DisabilityVerification',
            'gro-au:Document',
            'gro-us:VerificationDocumentation',
            'gro-au:AcademicSuitabilityEvidence',
            'gro-uk:EvidenceOfEligibility',
            'gro-uk:WrittenEvidence',
            'gro-us:VerificationDocument',
            'gro-ca:RemediationEvidenceOrPlan',
            'gro-us:StatementOfEducationalPurpose',
            'gro-au:Evidence',
            'gro-ca:Document',
            'gro-uk:OtherRulesDocument',
        ],
    },
    'RegulatoryAgreement': {
        'label': 'Regulatory Agreement',
        'definition': 'A contractual or formal agreement between regulated parties such as providers, employers, or institutions.',
        'members': [
            'gro-uk:EmployerAgreement',
            'gro-au:EmploymentContract',
            'gro-uk:Contract',
        ],
    },
    'SignedDeclaration': {
        'label': 'Signed Declaration',
        'definition': 'A signed document or statement attesting to facts or confirming information for regulatory purposes.',
        'members': [
            'gro-uk:SelfDeclaration',
            'gro-uk:SignedDeclaration',
            'gro-uk:EmployerCompetencyStatement',
            'gro-uk:SignedStatement',
        ],
    },
    'RegulatoryDecision': {
        'label': 'Regulatory Decision',
        'definition': 'A formal determination, approval, assessment, or decision issued by a regulatory authority or process.',
        'members': [
            'gro-uk:Review',
            'gro-uk:Approval',
            'gro-au:AssessmentResults',
            'gro-uk:Judgement',
            'gro-au:Approval',
            'gro-ca:NoticeOfAssessment',
            'gro-au:Assessment',
            'gro-ca:CourseApproval',
            'gro-au:Decision',
            'gro-ca:AppealDecision',
            'gro-uk:SufficientEvidence',
            'gro-au:ReconsiderationOfDecision',
            'gro-au:Confirmation',
            'gro-uk:ApprenticeCertificate',
            'gro-au:CommonwealthAssistanceNotice',
        ],
    },
}

# Get original Cluster 1 member metadata (for lookups)
mega_cluster_1 = next((c for c in skipped_for_size if c['size'] == 120), None)
member_lookup = {m['class']: m for m in mega_cluster_1['members']}

# Get next cluster_id
next_id = max(c['cluster_id'] for c in cluster_names) + 1

# Track assigned members for misc/excluded count
assigned_members = set()

# Create shared superclass record for each theme
for theme_name, theme_data in THEME_DECOMPOSITION.items():
    # Verify all members exist in original cluster
    valid_members_meta = []
    missing_members = []
    
    for member_qname in theme_data['members']:
        if member_qname in member_lookup:
            valid_members_meta.append(member_lookup[member_qname])
            assigned_members.add(member_qname)
        else:
            missing_members.append(member_qname)
    
    if missing_members:
        print(f'WARNING for theme {theme_name}: {len(missing_members)} members not found:')
        for m in missing_members:
            print(f'    - {m}')
    
    # Build jurisdictions list
    jurisdictions = sorted(set(m['jurisdiction'] for m in valid_members_meta))
    
    # Create record
    new_record = {
        'cluster_id':            next_id,
        'parent':                'cco:Resource',
        'jurisdictions':         jurisdictions,
        'size':                  len(valid_members_meta),
        'members':               valid_members_meta,
        'shared_class_name':     theme_name,
        'label':                 theme_data['label'],
        'definition':            theme_data['definition'],
        'naming_justification':  f'Manually identified theme from decomposition of 120-member transitive cluster into 7 semantically coherent sub-themes after manual review.',
        'status':                'OK_MANUAL_THEME',
    }
    cluster_names.append(new_record)
    print(f'Added theme: gro:{theme_name} ({len(valid_members_meta)} members, {len(jurisdictions)} jurisdictions)')
    next_id += 1

# Report unassigned members from original 120
unassigned = [m for m in mega_cluster_1['members'] if m['class'] not in assigned_members]
print(f'\nUnassigned (no theme): {len(unassigned)} members')
if unassigned:
    print('Unassigned members (kept as Stage 5B pairs only, no shared superclass):')
    for m in unassigned:
        print(f'  - {m["jurisdiction"]}: {m["class"]} — "{m["label"]}"')

# Final summary
print(f'\n{"="*60}')
print(f'Total shared superclasses now: {len(cluster_names)}')
status_counts = Counter(c['status'] for c in cluster_names)
for s, n in status_counts.most_common():
    print(f'  {s}: {n}')